# Quantization-error LoRA for text-to-CAD

## Purpose

Train a small open model to turn a plain-English part description into an OpenSCAD program, then check the program by compiling it and measuring the mesh. The part families are spacers, bolt flanges, and rectangular plates. The model is Qwen3-0.6B (Apache 2.0). Everything here is free: synthetic data, a Kaggle T4, no paid API.

The comparison is the point of the run. Same data, same step budget, same rank:

1. Untouched fp16 model, no training.
2. NF4-quantized base with a LoRA adapter that starts at zero, then trains.
3. The same adapter, but initialized with LoftQ so the frozen quantized base plus the adapter starts closer to the original weights.

A prediction counts only if OpenSCAD compiles it, the bounding box is within 1 mm, and the volume error is smaller than one missing hole.

## Steps

1. Build a disjoint train and test set with code, so the test parts never appear in training.
2. Confirm the GPU is a T4. This image has no kernels for a P100.
3. Install OpenSCAD and prove the checker on known-good and intentionally wrong programs.
4. Score the untouched model.
5. Train the random-init adapter and the quantization-error adapter.
6. Write `results.json`, the loss plot, and a short table.

## Budget

1000 training parts, 36 test parts, rank 8, 180 optimizer steps, LoftQ for 4 iterations. This is a short run on purpose: a long run lets both adapters catch up and hides the initialization.


In [1]:
# Run constants. Change these only if a rerun needs a smaller budget.
MODEL_ID = "Qwen/Qwen3-0.6B"
SEED = 7
TRAIN_N = 1000
EVAL_N = 36
RANK = 8
LORA_ALPHA = 8
LOFTQ_ITERS = 4
LR = 2e-4
OPT_STEPS = 180
BATCH = 1
ACCUM = 16
GEN_BATCH = 4
MAX_LEN_CAP = 512
MAX_NEW = 360
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
WORKDIR = "/kaggle/working"


## The task

Each example is one part. A spacer is a tube. A flange is a disc with a bore and a bolt circle. A plate is a rectangle with four corner holes. Holes are kept inside the part and clear of each other, so the volume of the solid minus the holes is exact.

The prompt already opens `<scad>`. The model writes the program and closes it with `</scad>`. Training masks the prompt, so the loss is only on the program.


In [2]:
import math
import random

SPACER_TEXT = [
    "Make a circular spacer. Outer diameter {outer} mm, bore {bore} mm, height {height} mm.",
    "A round spacer, {height} mm thick, outside diameter {outer} mm, center hole {bore} mm.",
    "Spacer ring: OD {outer} mm, ID {bore} mm, length {height} mm.",
]
FLANGE_TEXT = [
    "Circular flange, outer diameter {outer} mm, center bore {bore} mm, thickness {height} mm, {n} bolt holes of diameter {bolt} mm on a {bcd} mm bolt circle.",
    "Make a flange: OD {outer} mm, bore {bore} mm, {height} mm thick, {n} holes diameter {bolt} mm, bolt circle {bcd} mm.",
    "Flange plate, {outer} mm across, {bore} mm bore, height {height} mm, {n} bolts of {bolt} mm on a {bcd} mm circle.",
]
PLATE_TEXT = [
    "Rectangular plate, {length} mm by {width} mm, thickness {thickness} mm, four corner holes of diameter {hole} mm centered {margin} mm from each edge.",
    "A flat plate {length} x {width} x {thickness} mm with 4 corner holes, diameter {hole} mm, inset {margin} mm from the edges.",
    "Plate {length} mm long, {width} mm wide, {thickness} mm thick. Drill four {hole} mm holes, each center {margin} mm in from the edges.",
]


def spec_key(spec):
    return tuple(sorted(spec.items()))


def valid_spec(spec):
    kind = spec["family"]
    if kind == "spacer":
        return spec["outer"] >= spec["bore"] + 10 and spec["bore"] >= 8 and spec["height"] >= 6
    if kind == "flange":
        outer_r = spec["outer"] / 2
        bore_r = spec["bore"] / 2
        bolt_r = spec["bolt"] / 2
        radius = spec["bcd"] / 2
        chord = spec["bcd"] * math.sin(math.pi / spec["n"])
        inside = radius + bolt_r <= outer_r - 1.5
        outside_bore = radius - bolt_r >= bore_r + 1.5
        separated = chord >= spec["bolt"] + 2
        even_circle = spec["bcd"] % 2 == 0
        return inside and outside_bore and separated and even_circle and spec["height"] >= 8
    if kind == "plate":
        inset = spec["margin"] >= spec["hole"] / 2 + 1
        apart_l = spec["length"] - 2 * spec["margin"] >= spec["hole"]
        apart_w = spec["width"] - 2 * spec["margin"] >= spec["hole"]
        return inset and apart_l and apart_w and spec["thickness"] >= 6
    return False


def sample_spec(rng):
    for _ in range(200):
        kind = rng.choice(["spacer", "flange", "plate"])
        if kind == "spacer":
            spec = {
                "family": "spacer",
                "outer": rng.choice([24, 28, 30, 32, 36, 40, 44, 48, 52, 56]),
                "bore": rng.choice([8, 10, 12, 14, 16]),
                "height": rng.choice([6, 8, 10, 12, 14, 16]),
            }
        elif kind == "flange":
            spec = {
                "family": "flange",
                "outer": rng.choice([48, 52, 56, 60, 64, 68, 72, 76, 80]),
                "bore": rng.choice([8, 10, 12, 14, 16]),
                "height": rng.choice([8, 10, 12, 14]),
                "n": rng.choice([3, 4, 6]),
                "bolt": rng.choice([6, 8]),
                "bcd": rng.choice([28, 32, 36, 40, 44, 48, 52, 56, 60, 64, 68]),
            }
        else:
            spec = {
                "family": "plate",
                "length": rng.choice([36, 40, 44, 50, 56, 60, 70, 80]),
                "width": rng.choice([28, 32, 36, 40, 44, 50]),
                "thickness": rng.choice([6, 8, 10, 12]),
                "hole": rng.choice([6, 8]),
                "margin": rng.choice([8, 10, 12]),
            }
        if valid_spec(spec):
            return spec
    raise RuntimeError("could not sample a valid part")


def part_text(spec, rng):
    if spec["family"] == "spacer":
        template = rng.choice(SPACER_TEXT)
    elif spec["family"] == "flange":
        template = rng.choice(FLANGE_TEXT)
    else:
        template = rng.choice(PLATE_TEXT)
    return template.format(**spec)


def program_for(spec):
    lines = ["$fn = 64;", "difference() {"]
    if spec["family"] == "spacer":
        h = spec["height"]
        lines.append(f"  cylinder(h={h}, d={spec['outer']});")
        lines.append(f"  translate([0, 0, -1]) cylinder(h={h + 2}, d={spec['bore']});")
    elif spec["family"] == "flange":
        h = spec["height"]
        lines.append(f"  cylinder(h={h}, d={spec['outer']});")
        lines.append(f"  translate([0, 0, -1]) cylinder(h={h + 2}, d={spec['bore']});")
        for i in range(spec["n"]):
            angle = int(round(i * 360 / spec["n"]))
            lines.append(
                f"  rotate([0, 0, {angle}]) translate([{spec['bcd']}/2, 0, -1]) cylinder(h={h + 2}, d={spec['bolt']});"
            )
    else:
        t = spec["thickness"]
        length, width, margin, hole = spec["length"], spec["width"], spec["margin"], spec["hole"]
        lines.append(f"  cube([{length}, {width}, {t}]);")
        centers = [
            (margin, margin),
            (length - margin, margin),
            (margin, width - margin),
            (length - margin, width - margin),
        ]
        for x, y in centers:
            lines.append(f"  translate([{x}, {y}, -1]) cylinder(h={t + 2}, d={hole});")
    lines.append("}")
    return "\n".join(lines) + "\n"


def prompt_for(text):
    return (
        "Write an OpenSCAD program for the part below.\n"
        "Use only difference, cylinder, cube, translate, and rotate.\n"
        "The opening tag is already written. End with </scad>.\n"
        "Dimensions are millimeters.\n\n"
        f"Part: {text}\n"
        "<scad>\n"
    )


def analytical_volume(spec):
    if spec["family"] == "spacer":
        return math.pi / 4 * spec["height"] * (spec["outer"] ** 2 - spec["bore"] ** 2)
    if spec["family"] == "flange":
        base = math.pi / 4 * spec["height"] * (spec["outer"] ** 2 - spec["bore"] ** 2)
        holes = spec["n"] * math.pi / 4 * spec["height"] * spec["bolt"] ** 2
        return base - holes
    solid = spec["length"] * spec["width"] * spec["thickness"]
    holes = 4 * math.pi / 4 * spec["thickness"] * spec["hole"] ** 2
    return solid - holes


def smallest_feature_volume(spec):
    if spec["family"] == "spacer":
        return math.pi / 4 * spec["height"] * spec["bore"] ** 2
    if spec["family"] == "flange":
        bore = math.pi / 4 * spec["height"] * spec["bore"] ** 2
        bolt = math.pi / 4 * spec["height"] * spec["bolt"] ** 2
        return min(bore, bolt)
    return math.pi / 4 * spec["thickness"] * spec["hole"] ** 2


def make_split(train_n, eval_n, seed):
    train_rng = random.Random(seed)
    test_rng = random.Random(seed + 1)
    text_rng = random.Random(seed + 2)
    train, seen = [], set()
    attempts = 0
    while len(train) < train_n:
        attempts += 1
        if attempts > train_n * 100:
            raise RuntimeError(f"only found {len(train)} unique training parts")
        spec = sample_spec(train_rng)
        key = spec_key(spec)
        if key in seen:
            continue
        seen.add(key)
        text = part_text(spec, text_rng)
        train.append({"spec": spec, "text": text, "program": program_for(spec)})
    test = []
    attempts = 0
    while len(test) < eval_n:
        attempts += 1
        if attempts > eval_n * 200:
            raise RuntimeError(f"only found {len(test)} unique test parts")
        spec = sample_spec(test_rng)
        key = spec_key(spec)
        if key in seen:
            continue
        seen.add(key)
        text = part_text(spec, text_rng)
        test.append({"spec": spec, "text": text, "program": program_for(spec)})
    train_text = {row["text"] for row in train}
    overlap = [row["text"] for row in test if row["text"] in train_text]
    if overlap:
        raise RuntimeError(f"test text leaked into train: {overlap[:3]}")
    for row in train + test:
        if not valid_spec(row["spec"]):
            raise RuntimeError(f"invalid spec slipped through: {row['spec']}")
        if analytical_volume(row["spec"]) <= 0:
            raise RuntimeError(f"non-positive volume: {row['spec']}")
        for key, value in row["spec"].items():
            if key == "family":
                continue
            if str(value) not in row["text"]:
                raise RuntimeError(f"number {key}={value} missing from text: {row['text']}")
            if key == "n":
                if row["program"].count("rotate(") != value:
                    raise RuntimeError(f"flange hole count mismatch:\n{row['program']}")
                continue
            if str(value) not in row["program"]:
                raise RuntimeError(f"number {key}={value} missing from program:\n{row['program']}")
    return train, test


TRAIN_ROWS, TEST_ROWS = make_split(TRAIN_N, EVAL_N, SEED)
print(f"train {len(TRAIN_ROWS)}  test {len(TEST_ROWS)}")
print("example prompt:\n")
print(prompt_for(TRAIN_ROWS[0]["text"]))
print(TRAIN_ROWS[0]["program"] + "</scad>")
print("analytical volume", round(analytical_volume(TRAIN_ROWS[0]["spec"]), 1))


train 1000  test 36
example prompt:

Write an OpenSCAD program for the part below.
Use only difference, cylinder, cube, translate, and rotate.
The opening tag is already written. End with </scad>.
Dimensions are millimeters.

Part: A flat plate 36 x 44 x 8 mm with 4 corner holes, diameter 6 mm, inset 8 mm from the edges.
<scad>

$fn = 64;
difference() {
  cube([36, 44, 8]);
  translate([8, 8, -1]) cylinder(h=10, d=6);
  translate([28, 8, -1]) cylinder(h=10, d=6);
  translate([8, 36, -1]) cylinder(h=10, d=6);
  translate([28, 36, -1]) cylinder(h=10, d=6);
}
</scad>
analytical volume 11767.2


## Environment

The checker is OpenSCAD, run headless. Training libraries are installed without replacing the image's PyTorch, because a torch upgrade would drop the T4 kernels. The run stops immediately if the card is not a T4.


In [3]:
import os
import subprocess
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
os.environ["DEBIAN_FRONTEND"] = "noninteractive"

import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAP = torch.cuda.get_device_capability(0)
print("gpu", GPU_NAME, "capability", GPU_CAP, flush=True)
if GPU_CAP[0] < 7:
    raise RuntimeError(
        f"Need a T4 (capability 7.5). This session is {GPU_NAME} {GPU_CAP}. "
        "P100 cannot run the current Kaggle PyTorch build."
    )

def stream(cmd):
    print("+", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"command failed ({code}): {' '.join(cmd)}")

def stream_apt(args):
    try:
        stream(["apt-get", *args])
    except RuntimeError as first:
        print("apt-get failed, retrying with sudo", flush=True)
        print(first, flush=True)
        stream(["sudo", "-n", "apt-get", *args])

stream_apt(["update", "-qq"])
stream_apt(["install", "-y", "-qq", "openscad", "xvfb"])
stream([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade-strategy", "only-if-needed",
    "torchao>=0.16.0",
    "transformers>=4.51.3,<5",
    "peft>=0.15.0",
    "accelerate>=1.2.0",
    "bitsandbytes>=0.45.0",
    "safetensors>=0.4.5",
])

import torch as torch_after
import torchao
print("torch after install", torch_after.__version__, "cuda", torch_after.cuda.is_available(), flush=True)
print("torchao", torchao.__version__, flush=True)
if not torch_after.cuda.is_available():
    raise RuntimeError("CUDA disappeared after pip install")
print("openscad:", flush=True)
stream(["openscad", "--version"])


torch 2.11.0+cu128 cuda True
gpu Tesla T4 capability (7, 5)
+ apt-get update -qq
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
+ apt-get install -y -qq openscad xvfb
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading d

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


torch after install 2.11.0+cu128 cuda True
torchao 0.18.0
openscad:
+ openscad --version
OpenSCAD version 2021.01


## Checker

OpenSCAD compiles the program to an STL. The score uses that mesh, not a string match. A part passes when the bounding box matches the gold mesh within 1 mm and the absolute volume error is under 35% of the smallest hole. That bar is high enough to ignore facet error at `$fn = 64`, and low enough that deleting one hole fails.

The self-test compiles one part of each family, checks the mesh against the analytical volume, and checks that a program with the wrong outer size fails.


In [4]:
import shutil
import struct
import tempfile
from pathlib import Path

import numpy as np

OPENSCAD = shutil.which("openscad")
if OPENSCAD is None:
    raise RuntimeError("openscad is not on PATH after install")
USE_XVFB = False


def read_stl(path):
    data = Path(path).read_bytes()
    if len(data) < 84:
        raise RuntimeError(f"stl too small: {path}")
    count = struct.unpack_from("<I", data, 80)[0]
    binary_size = 84 + count * 50
    if binary_size == len(data):
        tris = np.empty((count, 3, 3), dtype=np.float64)
        for i in range(count):
            vals = struct.unpack_from("<12f", data, 84 + i * 50)
            tris[i] = np.array(vals[3:], dtype=np.float64).reshape(3, 3)
        return tris
    text = data.decode("utf-8", "ignore")
    verts = []
    cur = []
    for line in text.splitlines():
        parts = line.strip().split()
        if parts[:1] == ["vertex"] and len(parts) == 4:
            cur.append([float(parts[1]), float(parts[2]), float(parts[3])])
            if len(cur) == 3:
                verts.append(cur)
                cur = []
    if not verts:
        raise RuntimeError(f"could not parse stl: {path}")
    return np.array(verts, dtype=np.float64)


def mesh_metrics(tris):
    volume = 0.0
    for tri in tris:
        volume += float(np.dot(tri[0], np.cross(tri[1], tri[2])) / 6.0)
    pts = tris.reshape(-1, 3)
    dims = pts.max(axis=0) - pts.min(axis=0)
    return abs(volume), [float(x) for x in dims]


def compile_program(source, dest_stl=None):
    source = source.strip()
    if not source or len(source) > 6000 or "difference" not in source:
        return None, "rejected before compile"
    with tempfile.TemporaryDirectory() as tmp:
        scad_path = Path(tmp) / "part.scad"
        stl_path = Path(tmp) / "part.stl"
        scad_path.write_text(source + "\n", encoding="utf-8")
        cmd = [OPENSCAD, "-o", str(stl_path), str(scad_path)]
        if USE_XVFB:
            cmd = ["xvfb-run", "-a"] + cmd
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
        except subprocess.TimeoutExpired:
            return None, "openscad timeout"
        if proc.returncode != 0 or not stl_path.exists() or stl_path.stat().st_size < 84:
            err = (proc.stderr or proc.stdout or "compile failed")[-400:]
            return None, err
        try:
            metrics = mesh_metrics(read_stl(stl_path))
        except Exception as exc:
            return None, f"mesh parse failed: {exc}"
        if dest_stl:
            shutil.copyfile(stl_path, dest_stl)
        return metrics, None


def geometry_pass(gold, pred, spec):
    if gold is None or pred is None:
        return False
    _, gold_dims = gold
    pred_vol, pred_dims = pred
    gold_vol = gold[0]
    if any(abs(a - b) > 1.0 for a, b in zip(sorted(gold_dims), sorted(pred_dims))):
        return False
    tol = 0.35 * smallest_feature_volume(spec)
    return abs(pred_vol - gold_vol) <= tol


def prove_checker():
    global USE_XVFB
    shown = {"spacer": None, "flange": None, "plate": None}
    for row in TRAIN_ROWS:
        kind = row["spec"]["family"]
        if shown[kind] is None:
            shown[kind] = row
        if all(shown.values()):
            break
    probe = shown["spacer"]["program"]
    metrics, err = compile_program(probe)
    if metrics is None:
        print("headless openscad failed, retrying under xvfb", flush=True)
        print(err, flush=True)
        USE_XVFB = True
        metrics, err = compile_program(probe)
    if metrics is None:
        raise RuntimeError(f"openscad self-test could not compile a gold spacer:\n{err}")
    for kind, row in shown.items():
        metrics, err = compile_program(row["program"])
        if metrics is None:
            raise RuntimeError(f"gold {kind} failed to compile:\n{err}")
        vol, dims = metrics
        analytical = analytical_volume(row["spec"])
        rel = abs(vol - analytical) / analytical
        print(
            f"gold {kind}: mesh {vol:.1f}  analytical {analytical:.1f}  rel {rel:.4f}  dims {[round(d, 2) for d in dims]}",
            flush=True,
        )
        if rel > 0.04:
            raise RuntimeError(f"gold {kind} mesh volume is too far from the formula ({rel:.3f})")
        if not geometry_pass(metrics, metrics, row["spec"]):
            raise RuntimeError(f"gold {kind} does not pass against itself")
        spec = row["spec"]
        if spec["family"] == "plate":
            bad = row["program"].replace(f"cube([{spec['length']},", f"cube([{spec['length'] + 8},", 1)
        else:
            bad = row["program"].replace(f"d={spec['outer']}", f"d={spec['outer'] + 8}", 1)
        bad_metrics, bad_err = compile_program(bad)
        if bad_metrics is None:
            raise RuntimeError(f"tampered {kind} should still compile:\n{bad_err}")
        if geometry_pass(metrics, bad_metrics, spec):
            raise RuntimeError(f"tampered {kind} was accepted; the checker is too loose")
    print("checker self-test passed", flush=True)


prove_checker()


gold spacer: mesh 5420.0  analytical 5428.7  rel 0.0016  dims [24.0, 24.0, 16.0]
gold flange: mesh 38642.3  analytical 38704.4  rel 0.0016  dims [72.0, 72.0, 10.0]
gold plate: mesh 11768.7  analytical 11767.2  rel 0.0001  dims [36.0, 44.0, 8.0]
checker self-test passed


## Training setup

Both trained models freeze an NF4-dequantized copy of each target matrix and learn a rank-8 adapter. Random init quantizes the original matrix and leaves the adapter at zero, which is ordinary QLoRA. Quantization-error init runs four LoftQ iterations: quantize the residual, fit the adapter to what the quantizer could not represent, and repeat. The frozen base is then the quantizer's output, and the adapter holds that low-rank repair.

LoftQ is applied by this notebook, not by a library flag. A one-matrix check must show that the repaired weights are closer to the original than plain NF4 before any training starts. Reconstruction error is also logged on the real model, because a task-score tie can still hide a real difference in the init.


In [5]:
import gc
import json
import time
import traceback
from pathlib import Path

import bitsandbytes as bnb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList


def nf4_dequant(weight):
    tensor = weight.detach().float().contiguous().cuda()
    try:
        packed = bnb.functional.quantize_4bit(tensor, quant_type="nf4", compress_statistics=False)
    except TypeError:
        packed = bnb.functional.quantize_4bit(tensor, quant_type="nf4")
    if isinstance(packed, tuple):
        quant, state = packed[0], packed[1]
    else:
        quant, state = packed, packed.quant_state
    restored = bnb.functional.dequantize_4bit(quant, state).float().cpu()
    if restored.shape != weight.shape:
        raise RuntimeError(f"nf4 shape {tuple(restored.shape)} != {tuple(weight.shape)}")
    if not torch.isfinite(restored).all():
        raise RuntimeError("nf4 dequant produced a non-finite value")
    return restored


def low_rank(residual, rank):
    try:
        u, s, vh = torch.linalg.svd(residual.float().cuda(), full_matrices=False)
        u, s, vh = u.cpu(), s.cpu(), vh.cpu()
    except RuntimeError:
        u, s, vh = torch.linalg.svd(residual.float().cpu(), full_matrices=False)
    b = u[:, :rank] * s[:rank].unsqueeze(0)
    a = vh[:rank, :]
    return b, a


def loftq_factors(weight, rank, iters):
    out, inn = weight.shape
    a = torch.zeros(rank, inn)
    b = torch.zeros(out, rank)
    for _ in range(iters):
        quant = nf4_dequant(weight - b @ a)
        b, a = low_rank(weight.float() - quant, rank)
    quant = nf4_dequant(weight - b @ a)
    return quant, a, b


def prove_nf4_and_loftq():
    torch.manual_seed(0)
    weight = torch.randn(128, 64)
    plain = nf4_dequant(weight)
    quant, a, b = loftq_factors(weight, rank=4, iters=4)
    plain_err = (weight - plain).norm().item()
    loft_err = (weight - (quant + b @ a)).norm().item()
    print(f"tiny matrix  plain nf4 err {plain_err:.4f}  loftq err {loft_err:.4f}", flush=True)
    if loft_err >= plain_err:
        raise RuntimeError("LoftQ did not beat plain NF4 on the probe matrix")


prove_nf4_and_loftq()


def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    vocab = tokenizer.get_vocab()
    eos = tokenizer.eos_token
    pad = None
    for cand in ["<|fim_pad|>", "<|endoftext|>", "<|fim_middle|>"]:
        if cand in vocab and cand != eos:
            pad = cand
            break
    tokenizer.pad_token = pad or eos
    tokenizer.padding_side = "right"
    return tokenizer


def encode_row(tokenizer, row):
    prompt_ids = tokenizer(prompt_for(row["text"]), add_special_tokens=True).input_ids
    answer_ids = tokenizer(row["program"] + "</scad>\n", add_special_tokens=False).input_ids
    eos = [tokenizer.eos_token_id]
    input_ids = prompt_ids + answer_ids + eos
    labels = [-100] * len(prompt_ids) + answer_ids + eos
    if len(input_ids) > MAX_LEN_CAP:
        raise RuntimeError(f"example is {len(input_ids)} tokens, over the cap of {MAX_LEN_CAP}")
    return input_ids, labels


def collate(rows, pad_id):
    width = max(len(ids) for ids, _ in rows)
    input_ids, labels, mask = [], [], []
    for ids, labs in rows:
        pad = width - len(ids)
        input_ids.append(ids + [pad_id] * pad)
        labels.append(labs + [-100] * pad)
        mask.append([1] * len(ids) + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(mask, dtype=torch.long),
    }


def batches_from(encoded, pad_id):
    bucket = []
    for item in encoded:
        bucket.append(item)
        if len(bucket) == BATCH:
            yield collate(bucket, pad_id)
            bucket = []
    if bucket:
        yield collate(bucket, pad_id)


def adapter_scale(module):
    scaling = module.scaling
    if isinstance(scaling, dict):
        key = "default" if "default" in scaling else next(iter(scaling))
        return float(scaling[key])
    return float(scaling)


def apply_init(model, mode):
    done = 0
    plain_errs, fixed_errs = [], []
    started = time.time()
    for name, module in model.named_modules():
        if not hasattr(module, "lora_A") or not hasattr(module, "base_layer"):
            continue
        base = module.base_layer
        if not isinstance(base, nn.Linear):
            continue
        weight = base.weight.detach().float().cpu()
        if mode == "random":
            quant = nf4_dequant(weight)
            a = b = None
        elif mode == "quant_error":
            quant, a, b = loftq_factors(weight, RANK, LOFTQ_ITERS)
        else:
            raise RuntimeError(mode)
        plain = nf4_dequant(weight)
        plain_errs.append((weight - plain).norm().item() / weight.norm().clamp(min=1e-8).item())
        if a is None:
            fixed_errs.append(plain_errs[-1])
        else:
            fixed_errs.append((weight - (quant + b @ a)).norm().item() / weight.norm().clamp(min=1e-8).item())
        with torch.no_grad():
            base.weight.copy_(quant.to(device=base.weight.device, dtype=base.weight.dtype))
            if a is not None:
                scale = adapter_scale(module)
                adapter_name = "default" if "default" in module.lora_A else next(iter(module.lora_A))
                a_param = module.lora_A[adapter_name].weight
                b_param = module.lora_B[adapter_name].weight
                a_param.copy_(a.to(device=a_param.device, dtype=a_param.dtype))
                b_param.copy_((b / scale).to(device=b_param.device, dtype=b_param.dtype))
        done += 1
        if done % 20 == 0:
            print(f"  init {done} layers  {time.time() - started:.0f}s", flush=True)
            torch.cuda.empty_cache()
    if done == 0:
        suffixes = sorted({n.split(".")[-1] for n, m in model.named_modules() if isinstance(m, nn.Linear)})
        raise RuntimeError(f"no LoRA layers were initialized. linear suffixes: {suffixes}")
    stats = {
        "layers": done,
        "rel_plain": sum(plain_errs) / len(plain_errs),
        "rel_fixed": sum(fixed_errs) / len(fixed_errs),
    }
    print(
        f"init {mode}: layers {done}  rel nf4 {stats['rel_plain']:.6f}  rel after {stats['rel_fixed']:.6f}",
        flush=True,
    )
    return stats


def load_fp16():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
    )
    model.config.use_cache = True
    model.eval()
    return model


def load_lora(mode):
    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
    )
    config = LoraConfig(
        r=RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=TARGET_MODULES,
        use_rslora=False,
    )
    model = get_peft_model(model, config)
    stats = apply_init(model, mode)
    for name, param in model.named_parameters():
        param.requires_grad = "lora_" in name
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model.config.use_cache = False
    if mode == "quant_error" and not (stats["rel_fixed"] < stats["rel_plain"]):
        raise RuntimeError(f"LoftQ did not reduce reconstruction error: {stats}")
    return model, stats


def trainable(model):
    return [p for p in model.parameters() if p.requires_grad]


def mean_loss(model, encoded, pad_id, limit=8):
    model.eval()
    losses = []
    device = next(model.parameters()).device
    for batch in list(batches_from(encoded, pad_id))[:limit]:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            losses.append(model(**batch).loss.item())
    model.train()
    return sum(losses) / len(losses)


def train(model, encoded, pad_id):
    device = next(model.parameters()).device
    opt = torch.optim.AdamW(trainable(model), lr=LR)
    scaler = torch.amp.GradScaler("cuda")
    opt.zero_grad(set_to_none=True)
    pool = list(batches_from(encoded, pad_id))
    if not pool:
        raise RuntimeError("empty training set")
    cursor = 0
    opt_step = 0
    micro = 0
    losses = []
    tokens = 0
    started = time.time()
    model.train()
    while opt_step < OPT_STEPS:
        batch = {k: v.to(device) for k, v in pool[cursor].items()}
        cursor = (cursor + 1) % len(pool)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = model(**batch).loss / ACCUM
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss at micro-step {micro}")
        scaler.scale(loss).backward()
        tokens += int(batch["attention_mask"].sum().item())
        micro += 1
        if micro % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable(model), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            opt_step += 1
            losses.append(float(loss.item() * ACCUM))
            if opt_step % 20 == 0 or opt_step == 1:
                elapsed = time.time() - started
                mem = torch.cuda.max_memory_allocated() / 1024**3
                print(
                    f"step {opt_step}/{OPT_STEPS}  loss {losses[-1]:.4f}  "
                    f"tok/s {tokens / elapsed:.0f}  peak_mem {mem:.2f} GB",
                    flush=True,
                )
    return losses


class StopAtScad(StoppingCriteria):
    def __init__(self, tokenizer, prompt_width):
        self.tokenizer = tokenizer
        self.prompt_width = prompt_width

    def __call__(self, input_ids, scores, **kwargs):
        flags = []
        for row in input_ids:
            text = self.tokenizer.decode(row[self.prompt_width:], skip_special_tokens=True)
            flags.append("</scad>" in text)
        return torch.tensor(flags, device=input_ids.device)


def extract_program(text):
    text = text.replace("```openscad", "").replace("```", "")
    if "</scad>" in text:
        text = text.split("</scad>", 1)[0]
    if "<scad>" in text:
        text = text.split("<scad>", 1)[1]
    return text.strip() + "\n"


def generate_programs(model, tokenizer, rows):
    was_training = model.training
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    device = next(model.parameters()).device
    programs = []
    print("first prompt tail:", prompt_for(rows[0]["text"])[-180:].replace("\n", " | "), flush=True)
    for start in range(0, len(rows), GEN_BATCH):
        chunk = rows[start:start + GEN_BATCH]
        prompts = [prompt_for(row["text"]) for row in chunk]
        tokens = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN_CAP)
        width = tokens["input_ids"].shape[1]
        tokens = {k: v.to(device) for k, v in tokens.items()}
        with torch.inference_mode():
            gen = model.generate(
                **tokens,
                max_new_tokens=MAX_NEW,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                stopping_criteria=StoppingCriteriaList([StopAtScad(tokenizer, width)]),
            )
        for row_ids in gen:
            programs.append(extract_program(tokenizer.decode(row_ids[width:], skip_special_tokens=True)))
    tokenizer.padding_side = "right"
    model.config.use_cache = False
    print("first completion:\n" + programs[0][:700], flush=True)
    if was_training:
        model.train()
    return programs


def score_programs(rows, programs):
    compile_ok = 0
    passed = 0
    vol_errs = []
    examples = []
    for index, (row, program) in enumerate(zip(rows, programs)):
        gold, gold_err = compile_program(row["program"])
        pred, pred_err = compile_program(program)
        ok = pred is not None
        good = geometry_pass(gold, pred, row["spec"])
        compile_ok += int(ok)
        passed += int(good)
        if gold is not None and pred is not None and gold[0] > 0:
            vol_errs.append(abs(pred[0] - gold[0]) / gold[0])
        if index < 5:
            examples.append({
                "text": row["text"],
                "pass": good,
                "compiled": ok,
                "pred": program[:800],
                "error": None if pred_err is None else pred_err[-400:],
            })
    n = len(rows)
    return {
        "n": n,
        "compile_rate": compile_ok / n,
        "pass_rate": passed / n,
        "mean_rel_volume_error": None if not vol_errs else sum(vol_errs) / len(vol_errs),
        "examples": examples,
    }


def release(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


def save_json(name, obj):
    path = Path(WORKDIR) / name
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    print("wrote", path, flush=True)


tiny matrix  plain nf4 err 8.3800  loftq err 7.0792


## Runs

The untouched model is scored first and unloaded. Each trained run records loss before step 1, then trains, then compiles every test part. Peak memory is printed every 20 steps. Artifacts land in `/kaggle/working`: `results.json`, `progress.json`, `loss_curves.png`, and the two adapters.


In [6]:
def main():
    global MAX_NEW
    banner = lambda title: print("\n" + "=" * 60 + f"\n{title}\n" + "=" * 60, flush=True)
    t0 = time.time()
    banner("TOKENIZER")
    tokenizer = load_tokenizer()
    print("pad", tokenizer.pad_token, "eos", tokenizer.eos_token, flush=True)
    encoded = [encode_row(tokenizer, row) for row in TRAIN_ROWS]
    lengths = [len(ids) for ids, _ in encoded]
    answer_lens = [len(tokenizer(row["program"] + "</scad>\n", add_special_tokens=False).input_ids) for row in TRAIN_ROWS]
    MAX_NEW = max(answer_lens) + 16
    print(f"tokens prompt+answer max {max(lengths)}  mean {sum(lengths)/len(lengths):.0f}  max new {MAX_NEW}", flush=True)
    probe = list(batches_from(encoded[:8], tokenizer.pad_token_id))
    progress = {"gpu": GPU_NAME, "capability": list(GPU_CAP), "max_new": MAX_NEW}

    banner("UNTOUCHED FP16")
    model = load_fp16()
    programs = generate_programs(model, tokenizer, TEST_ROWS)
    untouched = score_programs(TEST_ROWS, programs)
    print("untouched", {k: untouched[k] for k in ("n", "compile_rate", "pass_rate")}, flush=True)
    progress["untouched_fp16"] = {k: untouched[k] for k in ("n", "compile_rate", "pass_rate", "mean_rel_volume_error")}
    save_json("progress.json", progress)
    release(model)

    banner("RANDOM INIT")
    model, random_stats = load_lora("random")
    pre_random = mean_loss(model, encoded, tokenizer.pad_token_id)
    print("random loss before training", round(pre_random, 4), flush=True)
    loss_random = train(model, encoded, tokenizer.pad_token_id)
    programs = generate_programs(model, tokenizer, TEST_ROWS)
    random_score = score_programs(TEST_ROWS, programs)
    print("random", {k: random_score[k] for k in ("n", "compile_rate", "pass_rate")}, flush=True)
    model.save_pretrained(str(Path(WORKDIR) / "adapter-random"))
    progress["random"] = {
        "init": random_stats,
        "loss_before": pre_random,
        "score": {k: random_score[k] for k in ("n", "compile_rate", "pass_rate", "mean_rel_volume_error")},
    }
    save_json("progress.json", progress)
    release(model)

    banner("QUANTIZATION-ERROR INIT")
    model, quant_stats = load_lora("quant_error")
    pre_quant = mean_loss(model, encoded, tokenizer.pad_token_id)
    print("quant-error loss before training", round(pre_quant, 4), flush=True)
    loss_quant = train(model, encoded, tokenizer.pad_token_id)
    programs = generate_programs(model, tokenizer, TEST_ROWS)
    quant_score = score_programs(TEST_ROWS, programs)
    print("quant-error", {k: quant_score[k] for k in ("n", "compile_rate", "pass_rate")}, flush=True)
    model.save_pretrained(str(Path(WORKDIR) / "adapter-quant-error"))
    release(model)

    plt.figure(figsize=(6.2, 4))
    plt.plot(loss_random, label="random init")
    plt.plot(loss_quant, label="quant-error init")
    plt.xlabel("optimizer step")
    plt.ylabel("train loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(Path(WORKDIR) / "loss_curves.png"), dpi=120)
    plt.close()

    results = {
        "model": MODEL_ID,
        "gpu": GPU_NAME,
        "train_n": TRAIN_N,
        "eval_n": EVAL_N,
        "opt_steps": OPT_STEPS,
        "rank": RANK,
        "loftq_iters": LOFTQ_ITERS,
        "seconds": time.time() - t0,
        "loss_before": {"random": pre_random, "quant_error": pre_quant},
        "init": {"random": random_stats, "quant_error": quant_stats},
        "scores": {
            "untouched_fp16": untouched,
            "random_lora": random_score,
            "quant_error_lora": quant_score,
        },
        "loss_random": loss_random,
        "loss_quant_error": loss_quant,
    }
    save_json("results.json", results)
    save_json("progress.json", {"done": True, "seconds": results["seconds"]})

    print("\ncondition            compile    pass     vol_err", flush=True)
    for name, score in results["scores"].items():
        vol = score["mean_rel_volume_error"]
        vol_s = "n/a" if vol is None else f"{vol:.3f}"
        print(f"{name:20} {score['compile_rate']:.3f}     {score['pass_rate']:.3f}    {vol_s}", flush=True)
    print(f"elapsed_min {results['seconds']/60:.1f}", flush=True)
    print("PIPELINE_OK", flush=True)


try:
    main()
except Exception:
    traceback.print_exc()
    print("PIPELINE_FAIL", flush=True)
    raise



TOKENIZER


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

first prompt tail: opening tag is already written. End with </scad>. | Dimensions are millimeters. |  | Part: Flange plate, 60 mm across, 14 mm bore, height 8 mm, 4 bolts of 6 mm on a 28 mm circle. | <scad> | 


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


first completion:


untouched {'n': 36, 'compile_rate': 0.0, 'pass_rate': 0.0}
wrote /kaggle/working/progress.json

RANDOM INIT
  init 20 layers  1s
  init 40 layers  1s
  init 60 layers  2s
  init 80 layers  2s
  init 100 layers  3s
  init 120 layers  3s
  init 140 layers  4s
  init 160 layers  5s
  init 180 layers  5s
init random: layers 196  rel nf4 0.092996  rel after 0.092996
random loss before training 1.0887
step 1/180  loss 2.4450  tok/s 429  peak_mem 2.52 GB
step 20/180  loss 0.0288  tok/s 420  peak_mem 2.52 GB
step 40/180  loss 0.0145  tok/s 434  peak_mem 2.52 GB
step 60/180  loss 0.0001  tok/s 445  peak_mem 2.52 GB
step 80/180  loss 0.0000  tok/s 441  peak_mem 2.52 GB
step 100/180  loss 0.0000  tok/s 441  peak_mem 2.52 GB
step 120/180  loss 0.0000  tok/s 445  peak_mem 2.52 GB
step 140/180  loss 0.0001  tok/s 443  peak_mem 2.52 GB
step 160/180  loss 0.0000  tok/s 444  peak_mem 2.52 GB
step 180/180  loss 0.0000  tok/s 447  peak_mem 2.52 GB
first prompt tail: opening tag is alr